# How much forest was cleared at Muhos?

In 2026 Google cleared forest for a planned datacentre near Muhos in northern Finland. The Finnish Licensing and Supervision Authority (LVV) ordered the work to stop in October 2026, and the case is being investigated. The Guardian reports an area of about **330 ha**, roughly **3 km long and 2 km wide**, about 400 m from the Oulujoki river.

In this notebook we measure the cleared area ourselves from Sentinel-2. In Assignment 3 you downloaded whole products and processed them on your own computer. Here we only *describe* the processing in Python. The Copernicus Data Space Ecosystem runs it on its own servers.

The steps are
1. connect to the openEO service of CDSE
2. load Sentinel-2 L2A for a small box around the site
3. remove clouds and shadows with the scene classification (SCL)
4. make an NDVI composite for summer 2025 and one for autumn 2026
5. find the pixels where NDVI dropped strongly and count the hectares

In [ ]:
import os, importlib.util

proj_data = os.path.join(os.path.dirname(importlib.util.find_spec("rasterio").origin), "proj_data")
if os.path.isdir(proj_data):
    os.environ["PROJ_DATA"] = os.environ["PROJ_LIB"] = proj_data

import openeo
import numpy as np
import rasterio
import matplotlib.pyplot as plt

## 1. Connect

The first time you run this, a link opens where you log in with your CDSE account. Every account gets free monthly openEO credits, and this small analysis uses only a few.

In [ ]:
connection = openeo.connect("openeo.dataspace.copernicus.eu").authenticate_oidc()

## 2. The area and the data

The box is about 8 by 8 km around the cleared area, centred near 64.836 N, 26.114 E, east of Muhos.
We request the red band (B04), the near infrared band (B08) and the scene classification (SCL) from June 2025 to October 2026.
Nothing is downloaded yet. `load_collection` only describes what we want.

In [ ]:
area = {"west": 26.03, "south": 64.80, "east": 26.20, "north": 64.87, "crs": "EPSG:4326"}

cube = connection.load_collection(
    "SENTINEL2_L2A",
    spatial_extent=area,
    temporal_extent=["2025-06-01", "2026-11-01"],
    bands=["B04", "B08", "SCL"],
    max_cloud_cover=60,
)

## 3. Remove clouds and shadows

Exactly as in Assignment 3 we keep only the SCL classes 4 (vegetation) and 5 (not vegetated). Everything else, clouds, shadows, water, snow, becomes no data. Here the masking runs on the server, for every image in the period at once.

In [ ]:
scl = cube.band("SCL")
remove = ~((scl == 4) | (scl == 5))          # True for the pixels we do not want
clean = cube.filter_bands(["B04", "B08"]).mask(remove)

## 4. NDVI composites before and after

For each period we take the **median** NDVI of all clear observations of each pixel. The median ignores the odd cloud that the SCL missed.

- **Before** is June to August 2025, when the forest still stood.
- **After** is September to October 2026, after the felling.

In [ ]:
ndvi = clean.ndvi(nir="B08", red="B04")

before = ndvi.filter_temporal("2025-06-01", "2025-09-01").reduce_dimension(dimension="t", reducer="median")
after = ndvi.filter_temporal("2026-09-01", "2026-11-01").reduce_dimension(dimension="t", reducer="median")
change = after - before

## 5. Which pixels were cleared?

A pixel counts as cleared when it was green before (NDVI above 0.6) and its NDVI fell by more than 0.3. Feel free to try other thresholds and see how the area changes.

In [ ]:
cleared = (before > 0.6) & (change < -0.3)

## 6. Run it and download only the result

Only now does the server do the work. We download four small GeoTIFFs of the box, a few megabytes in total, instead of dozens of full Sentinel-2 products.

In [ ]:
before.download("muhos_ndvi_before.tif")
after.download("muhos_ndvi_after.tif")
change.download("muhos_ndvi_change.tif")
cleared.download("muhos_cleared.tif")

## 7. Count the hectares

In [ ]:
with rasterio.open("muhos_cleared.tif") as src:
    mask = src.read(1)
    crs = src.crs.to_string()
    pixel_w, pixel_h = src.res
    pixel_area = abs(src.transform.a * src.transform.e)   # square metres per pixel

cleared_pixels = int(np.nansum(mask == 1))
hectares = cleared_pixels * pixel_area / 10000

print(f"Coordinate system    {crs}")
print(f"Pixel size           {pixel_w:.0f} m x {pixel_h:.0f} m")
print(f"Cleared pixels       {cleared_pixels:,}")
print(f"Cleared area in box  {hectares:.0f} ha")

The number counts **every** strong NDVI drop in the box. If there are other clear-cuts nearby they are included too, so look at the map in the next cell and check that the large block is the site.

In [ ]:
def show(path, ax, title, **kw):
    with rasterio.open(path) as src:
        a = src.read(1).astype(float)
    im = ax.imshow(a, **kw)
    ax.set_title(title); ax.axis("off")
    return im

fig, axes = plt.subplots(1, 4, figsize=(18, 5))
show("muhos_ndvi_before.tif", axes[0], "NDVI summer 2025", cmap="RdYlGn", vmin=-0.2, vmax=1)
show("muhos_ndvi_after.tif", axes[1], "NDVI autumn 2026", cmap="RdYlGn", vmin=-0.2, vmax=1)
im = show("muhos_ndvi_change.tif", axes[2], "Change", cmap="RdBu", vmin=-0.8, vmax=0.8)
fig.colorbar(im, ax=axes[2], shrink=0.7)
show("muhos_cleared.tif", axes[3], "Cleared pixels", cmap="Greys", vmin=0, vmax=1)
plt.tight_layout()
plt.show()

## 8. Google Cut

The map of cleared pixels also shows other clear-cuts and some fields around the site. We keep only the largest connected area, which is the Google block.

Inside the block a few patches dropped a little less than our threshold of 0.3. They lie inside the cleared site, so we fill these holes and count them as part of the block.

In [ ]:
from scipy import ndimage

with rasterio.open("muhos_cleared.tif") as src:
    cleared_all = np.nan_to_num(src.read(1)) == 1
    mask_profile = src.profile
    pixel_area = abs(src.transform.a * src.transform.e)


opened = ndimage.binary_opening(cleared_all, iterations=2)
labels, n = ndimage.label(opened)
sizes = ndimage.sum(opened, labels, range(1, n + 1))
largest = labels == (np.argmax(sizes) + 1)
block = ndimage.binary_fill_holes(largest)

print(f"Separate cleared areas in the box   {n}")
print(f"Largest area                        {largest.sum() * pixel_area / 10000:.0f} ha")
print(f"Largest area with its holes filled  {block.sum() * pixel_area / 10000:.0f} ha")

mask_profile.update(dtype="uint8", nodata=0, count=1)
with rasterio.open("muhos_block.tif", "w", **mask_profile) as dst:
    dst.write(block.astype("uint8"), 1)

with rasterio.open("muhos_ndvi_change.tif") as src:
    change_map = src.read(1)

plt.figure(figsize=(7, 7))
plt.imshow(change_map, cmap="RdBu", vmin=-0.8, vmax=0.8)
plt.contour(block, levels=[0.5], colors="magenta", linewidths=1.5)
plt.title("NDVI change and the Google block")
plt.axis("off")
plt.show()

## 9. What stood there? Luke's forest inventory

Sentinel-2 tells us **where** the forest disappeared, but not **how much wood** stood there. For that we use the Multi-Source National Forest Inventory (MS-NFI) of the Natural Resources Institute Finland (Luke). It maps the whole of Finland in 16 m pixels, with the wood volume of each tree species, the trunk diameter and much more. The 2023 maps describe the forest on 31 July 2023, before the felling.

Luke offers the maps through a web service (WCS), so we download only our small box, about half a megabyte per layer. The data are free under the CC BY 4.0 licence, ©Natural Resources Institute Finland 2023.

The layer names are Finnish. *manty* is pine, *kuusi* spruce, *koivu* birch, *muulp* other broadleaved trees, *tukki* saw logs, *kuitu* pulpwood, *ppa* basal area and *keskilapimitta* mean diameter. The ending *_1923* means the 2023 maps, made with field plots from 2019 to 2023.

In [ ]:
import requests
from rasterio.warp import transform_bounds

LUKE_WCS = "https://kartta.luke.fi/geoserver/MVMI/wcs"
LAYERS = ["manty", "kuusi", "koivu", "muulp",                  # volume of pine, spruce, birch, other broadleaved
          "mantytukki", "mantykuitu", "kuusitukki", "kuusikuitu", # saw logs (tukki) and pulpwood (kuitu)
          "koivutukki", "koivukuitu", "muulptukki", "muulpkuitu",
          "ppa", "keskilapimitta"]                             # basal area and mean trunk diameter

# The box of our block in the Finnish national grid, snapped outwards to Luke's 16 m pixels
with rasterio.open("muhos_block.tif") as src:
    west, south, east, north = transform_bounds(src.crs, "EPSG:3067", *src.bounds)
west, south = np.floor(west / 16) * 16, np.floor(south / 16) * 16
east, north = np.ceil(east / 16) * 16, np.ceil(north / 16) * 16

for name in LAYERS:
    r = requests.get(LUKE_WCS, params=[
        ("service", "WCS"), ("version", "2.0.1"), ("request", "GetCoverage"),
        ("coverageId", f"MVMI__{name}_1923"),
        ("subset", f"E({west:.0f},{east:.0f})"), ("subset", f"N({south:.0f},{north:.0f})"),
        ("format", "image/tiff")], timeout=120)
    r.raise_for_status()
    with open(f"luke_{name}.tif", "wb") as f:
        f.write(r.content)
    print(f"luke_{name}.tif  {len(r.content) / 1e6:.2f} MB")

## 10. Put the block on Luke's grid

Our block is in 10 m pixels in UTM zone 35N, Luke's maps are in 16 m pixels in the Finnish national grid (ETRS-TM35FIN). We reproject the block onto Luke's grid.

In [ ]:
from rasterio.warp import reproject
from rasterio.enums import Resampling

def read_luke(name):
    with rasterio.open(f"luke_{name}.tif") as src:
        a = src.read(1).astype(float)
    a[a == 255] = 0      # 255 means no forest land
    return a

with rasterio.open("luke_manty.tif") as luke:
    luke_transform, luke_crs, luke_shape = luke.transform, luke.crs, (luke.height, luke.width)
    luke_pixel_ha = abs(luke.transform.a * luke.transform.e) / 10000   # 0.0256 ha

# For every 16 m Luke pixel, the share of it that lies inside the cleared block (0 to 1)
with rasterio.open("muhos_block.tif") as src:
    share = np.zeros(luke_shape, dtype="float32")
    reproject(src.read(1).astype("float32"), share,
              src_transform=src.transform, src_crs=src.crs,
              dst_transform=luke_transform, dst_crs=luke_crs, resampling=Resampling.average)

print(f"Cleared block on Luke's grid {np.nansum(share) * luke_pixel_ha:.0f} ha")

## 11. A look at what we downloaded

Here are six of the Luke layers for our box, with the Google block outlined in magenta. White pixels are not forest land, such as fields, the river and roads. What kind of forest stood inside the block?

In [ ]:
layers = {
    "manty": ("Pine, m3/ha", 0, 160),
    "kuusi": ("Spruce, m3/ha", 0, 160),
    "koivu": ("Birch, m3/ha", 0, 80),
    "muulp": ("Other broadleaved, m3/ha", 0, 20),
    "ppa": ("Basal area, m2/ha", 5, 30),
    "keskilapimitta": ("Mean trunk diameter, cm", 5, 25),
}

fig, axes = plt.subplots(2, 3, figsize=(15, 10))

for ax, (name, (title, low, high)) in zip(axes.flat, layers.items()):
    with rasterio.open(f"luke_{name}.tif") as src:
        layer = src.read(1, masked=True)
    im = ax.imshow(layer, cmap="YlGn", vmin=low, vmax=high)
    ax.contour(share, levels=[0.5], colors="magenta", linewidths=1)
    fig.colorbar(im, ax=ax, shrink=0.7)
    ax.set_title(title)
    ax.axis("off")

plt.tight_layout()
plt.show()

## 12. How much wood was felled?

Each Luke pixel tells how many cubic metres of wood stood on one hectare. We multiply by the cleared share of the pixel and its area (16 m by 16 m is 0.0256 ha) and add everything up, for each species and split into saw logs and pulpwood.

The total is the sum of the four species. The web service also has a total volume layer, but it stores the values halved to fit them into 8 bits, so the sum of the species is safer.

In [ ]:
SPECIES = [("Pine", "manty"), ("Spruce", "kuusi"), ("Birch", "koivu"), ("Other broadleaved", "muulp")]

def total(name):
    """Cubic metres of one layer inside the cleared block."""
    return np.nansum(read_luke(name) * share * luke_pixel_ha)

print(f"{'':18}{'total':>9}{'saw logs':>10}{'pulpwood':>10}   m3")
for label, sp in SPECIES:
    print(f"{label:18}{total(sp):9.0f}{total(sp + 'tukki'):10.0f}{total(sp + 'kuitu'):10.0f}")

wood = sum(total(sp) for _, sp in SPECIES)
area_ha = np.nansum(share) * luke_pixel_ha
print(f"\nAll wood {wood:,.0f} m3 on {area_ha:.0f} ha, {wood / area_ha:.0f} m3 per ha")

## 13. How many trees?

The Guardian writes that about a quarter of a million trees stood on the 330 ha. Luke does not count trees, but it gives the **basal area**, the total cross-section of all trunks per hectare, and the **mean trunk diameter**. Dividing the first by the cross-section of one average trunk gives a rough number of trees per hectare.

It is only an estimate. The mean diameter leans towards the big trees, so small trees are undercounted.

In [ ]:
basal_area = read_luke("ppa")                    # m2 of trunk cross-section per ha
diameter_m = read_luke("keskilapimitta") / 100    # mean trunk diameter, cm to m
one_trunk = np.pi * (diameter_m / 2) ** 2         # cross-section of one average trunk, m2
with np.errstate(invalid="ignore", divide="ignore"):
    trees_per_ha = np.where(one_trunk > 0, basal_area / one_trunk, 0)

trees = np.nansum(trees_per_ha * share * luke_pixel_ha)
print(f"Our estimate   {trees:9,.0f} trees  ({trees / area_ha:.0f} per ha)")
print(f"The Guardian   {250000:9,.0f} trees  ({250000 / 330:.0f} per ha)")

In [ ]:
vol = sum(read_luke(sp) for _, sp in SPECIES)

fig, ax = plt.subplots(figsize=(7, 7))
im = ax.imshow(vol, cmap="YlGn", vmin=0, vmax=300)
ax.contour(share, levels=[0.5], colors="magenta", linewidths=1.5)
fig.colorbar(im, ax=ax, shrink=0.7, label="Wood volume in 2023, m3/ha")
ax.set_title("Luke forest inventory 2023 and the Google block")
ax.axis("off")
plt.show()